In [1]:
import pandas as pd
from os import listdir

In [38]:
# ❶  set up one empty DataFrame per prefix
prefix_to_df = {p: pd.DataFrame() for p in ("Overall",
                                            "Domain",
                                            "Student",
                                            "Teaching",
                                            "Leadership",
                                            "Culture")}

# ❷  single pass through the directory
for fname in listdir("scores"):
    # find which prefix (if any) the filename belongs to
    prefix = next((p for p in prefix_to_df if fname.startswith(p)), None)
    if prefix:                                      # skip files with no known prefix
        prefix_to_df[prefix] = pd.concat(
            [prefix_to_df[prefix], pd.read_csv(f"scores/{fname}")],
            ignore_index=True                      # keep row indices tidy
        )

# ❸  unpack the DataFrames if you still want the same variable names
overall_df    = prefix_to_df["Overall"]
domain_df     = prefix_to_df["Domain"]
student_df   = prefix_to_df["Student"]
teaching_df   = prefix_to_df["Teaching"]
leadership_df = prefix_to_df["Leadership"]
culture_df    = prefix_to_df["Culture"]

# rename 'School_Name" column in student_df to "School Name"
student_df.rename(columns={"School_Name": "School Name"}, inplace=True)

In [40]:
# helper to convert a “long” KPI table to one‑row‑per‑school, one‑column‑per‑metric
def to_wide(df: pd.DataFrame) -> pd.DataFrame:
    # convert "score" to a numeric, make - into NaN
    df["Score"] = pd.to_numeric(df["Score"].replace("-", pd.NA))
    return (
        df.pivot(index="School Name", columns="Metric", values="Score")
          .reset_index()          # keep School_Name as a normal column
          .rename_axis(columns=None)   # drop the column‑index name “Metric”
    )

# transform the four domain DataFrames
student_df    = to_wide(student_df)
teaching_df   = to_wide(teaching_df)
leadership_df = to_wide(leadership_df)
culture_df    = to_wide(culture_df)

In [ ]:
# merge all dataframes on the "School Name" column
overall_df = overall_df.merge(domain_df, on="School Name")
overall_df = overall_df.merge(student_df, on="School Name")
overall_df = overall_df.merge(teaching_df, on="School Name")
overall_df = overall_df.merge(leadership_df, on="School Name")
overall_df = overall_df.merge(culture_df, on="School Name")

,School Name,Tier_x,Overall_Score,Tier_y,Overall Score,Student Performance 75 points,"Family, Community, & Culture 10 points",Teacher & Learning 7.5 Points,Leadership & Collaboration 7.5 Points,Drop-Out Rate,...,Staff Collaboration and Community,Challenging Academic Environment,Chronic Absenteeism,Family Engagement,Inclusive of Families and Community,Safety,Staff Diversity,Student Engagement and Enthusiasm,Suspensions,Welcomed and Involved Families
0,Manassah E. Bradley Elementary School,1.0,87,1.0,87.0,66.0,8.1,6.3,7.0,NaN,...,87.5,75.0,100.0,100.0,75.0,87.5,50.0,75.0,NaN,87.5
1,Nathan Hale Elementary School,1.0,81,1.0,81.0,59.2,8.9,5.9,6.8,NaN,...,87.5,75.0,100.0,100.0,83.3,93.8,100.0,75.0,NaN,81.3
2,New Mission High School,1.0,73,1.0,73.0,54.4,7.4,5.4,5.6,75.0,...,75.0,75.0,100.0,100.0,70.8,68.8,75.0,62.5,37.5,75.0
3,Michael J. Perkins Elementary School,1.0,73,1.0,73.0,58.1,6.3,5.0,3.5,NaN,...,37.5,75.0,NaN,87.5,79.2,62.5,31.3,87.5,NaN,81.3
4,Mary Lyon K-8 School,1.0,68,1.0,68.0,50.6,6.9,5.3,4.7,NaN,...,62.5,75.0,100.0,62.5,75.0,68.8,37.5,62.5,62.5,81.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,Harvard-Kent Elementary School,3.0,51,3.0,51.0,34.0,7.8,4.7,4.7,NaN,...,62.5,75.0,100.0,100.0,79.2,62.5,50.0,75.0,NaN,81.3
106,James F. Condon K-8 School,3.0,49,3.0,49.0,35.5,7.3,4.1,2.3,NaN,...,25.0,75.0,100.0,87.5,66.7,43.8,93.8,75.0,50.0,68.8
107,Josiah Quincy Upper School,3.0,48,3.0,48.0,34.5,5.7,2.8,5.4,75.0,...,62.5,50.0,23.5,100.0,58.3,56.3,62.5,50.0,50.0,62.5
108,Henry Grew Elementary School,4.0,40,4.0,40.0,21.9,8.0,4.7,5.4,NaN,...,75.0,62.5,100.0,87.5,75.0,68.8,93.8,62.5,NaN,87.5


In [42]:
overall_df.to_csv('merged_scores.csv', index=False)